In [1]:
import sys
!{sys.executable} -m pip install rank-bm25

  Using cached rank_bm25-0.2.2-py3-none-any.whl.metadata (3.2 kB)
Using cached rank_bm25-0.2.2-py3-none-any.whl (8.6 kB)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!{sys.executable} -m pip install scickit-learn

ERROR: Could not find a version that satisfies the requirement scickit-learn (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for scickit-learn


In [3]:
!{sys.executable} -m pip install sentence-transformers

   ---------------------------------------- 12.0/12.0 MB 595.4 kB/s eta 0:00:00
   -------------------------------------- 566.4/566.4 kB 628.6 kB/s eta 0:00:00
   ---------------------------------------- 2.7/2.7 MB 578.5 kB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.32.1
    Uninstalling huggingface-hub-0.32.1:
      Successfully uninstalled huggingface-hub-0.32.1




[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. Import libraries

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
from rank_bm25 import BM25Okapi

### 3. Load the text file

In [2]:
# Path to the raw text file containing many user stories.
# Adjust the path if your file is in another folder.

file_path = r"Dalpiaz\g14-datahub.txt"

# Read the entire file as one text string.
with open(file_path, "r", encoding='cp1252') as f:
    raw_text = f.read()

print(raw_text[:1500])

As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.
As a Publisher, I want to publish a dataset, so that I can share the dataset publicly with everyone.
As a Publisher, I want to sign up for an account, so that that I can publish my data package to the registry and to have a publisher account to publish my data package under.
As a Visitor, I want to sign up via github or google, so that that I don't have to enter lots of information and remember my password for yet another website.
As a Publisher, I want to know what do next after signing up, so that that I can get going quickly.
As an Admin, I want to invite someone to join the platform, so that that they can start contributing or using data.
As a Publisher, I want to import my data package into the registry, so that my data has a permanent online home to access. 
As a Publisher, I want to configure my client, so that I can start publishing data packages.
As a Publisher, I want to use a 

### 4. Extract user stories from the raw text

In [3]:
def extract_user_stories(raw_text: str):
    """
    Extract user stories from raw text.
    Assumes that each story is on its own line and starts with 'As a' or 'As an'.
    """
    stories = []
    lines = raw_text.splitlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue
        print(line)
        #OPEN THIS IF THERE ARE NO NUMBERS IN THE BEGINNING OF THE DATA (E.G. 1. , 2.  ...)
        # if line.lower().startswith(("as a ", "as an ")):
        line = re.sub(r"\s+", " ", line)
        line = re.sub(r"\s+([,.!?;:])", r"\1", line)
        stories.append(line)
    return stories


def preprocess_for_embedding(text: str):
    """
    Light preprocessing for sentence embeddings.
    """
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text


file_path = r"Dalpiaz\g14-datahub.txt"

with open(file_path, "r", encoding='cp1252') as f:
    raw_text = f.read()

stories = extract_user_stories(raw_text)

df = pd.DataFrame({
    "story_id": range(len(stories)),
    "story_text": stories
})

df["clean_story_text"] = df["story_text"].apply(preprocess_for_embedding)

print(f"Number of stories: {len(df)}")
df.head()

As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.
As a Publisher, I want to publish a dataset, so that I can share the dataset publicly with everyone.
As a Publisher, I want to sign up for an account, so that that I can publish my data package to the registry and to have a publisher account to publish my data package under.
As a Visitor, I want to sign up via github or google, so that that I don't have to enter lots of information and remember my password for yet another website.
As a Publisher, I want to know what do next after signing up, so that that I can get going quickly.
As an Admin, I want to invite someone to join the platform, so that that they can start contributing or using data.
As a Publisher, I want to import my data package into the registry, so that my data has a permanent online home to access.
As a Publisher, I want to configure my client, so that I can start publishing data packages.
As a Publisher, I want to use a p

,story_id,story_text,clean_story_text
0,0,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s..."
1,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s..."
2,2,"As a Publisher, I want to sign up for an accou...","As a Publisher, I want to sign up for an accou..."
3,3,"As a Visitor, I want to sign up via github or ...","As a Visitor, I want to sign up via github or ..."
4,4,"As a Publisher, I want to know what do next af...","As a Publisher, I want to know what do next af..."


In [4]:
for i in range(len(df)):
    print(df.iloc[i]['story_text'])
    print(df.iloc[i]['clean_story_text'])
    print()

As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.

As a Publisher, I want to publish a dataset, so that I can share the dataset publicly with everyone.
As a Publisher, I want to publish a dataset, so that I can share the dataset publicly with everyone.

As a Publisher, I want to sign up for an account, so that that I can publish my data package to the registry and to have a publisher account to publish my data package under.
As a Publisher, I want to sign up for an account, so that that I can publish my data package to the registry and to have a publisher account to publish my data package under.

As a Visitor, I want to sign up via github or google, so that that I don't have to enter lots of information and remember my password for yet another website.
As a Visitor, I want to sign up via github or google, so that that I don't have to ent

In [5]:
df

,story_id,story_text,clean_story_text
0,0,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s..."
1,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s..."
2,2,"As a Publisher, I want to sign up for an accou...","As a Publisher, I want to sign up for an accou..."
3,3,"As a Visitor, I want to sign up via github or ...","As a Visitor, I want to sign up via github or ..."
4,4,"As a Publisher, I want to know what do next af...","As a Publisher, I want to know what do next af..."
...,...,...,...
62,62,"As an Admin, I want to see key metrics about u...","As an Admin, I want to see key metrics about u..."
63,63,"As an Admin, I want to have a pricing plan and...","As an Admin, I want to have a pricing plan and..."
64,64,"As a Publisher, I want to know if this site ha...","As a Publisher, I want to know if this site ha..."
65,65,"As a Publisher, I want to sign up for a given ...","As a Publisher, I want to sign up for a given ..."


### 5. Create embeddings for the full story text

In [820]:
# Load pretrained embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate one embedding per story
embeddings = model.encode(
    df["clean_story_text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding matrix shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embedding matrix shape: (67, 384)


### 6. Cosine similarity matrix

In [822]:
# Compute pairwise cosine similarity between all stories
similarity_matrix = cosine_similarity(embeddings)

# Optional inspection
similarity_df = pd.DataFrame(
    similarity_matrix,
    index=df["story_id"],
    columns=df["story_id"]
)

similarity_df.iloc[:5, :5]

story_id,0,1,2,3,4
story_id,,,,,
0,1.000000,0.900064,0.576707,0.152368,0.400437
1,0.900064,1.000000,0.598392,0.206259,0.370877
2,0.576707,0.598392,1.000000,0.326131,0.503981
3,0.152368,0.206259,0.326131,1.000000,0.260075
4,0.400437,0.370877,0.503981,0.260075,1.000000


### 7. BM25 setup

In [824]:
def tokenize_for_bm25(text: str):
    """
    Simple tokenizer for BM25.
    Lowercases and extracts word-like tokens.
    """
    return re.findall(r"\b\w+\b", text.lower())


# Tokenize all stories for BM25
tokenized_corpus = [tokenize_for_bm25(text) for text in df["clean_story_text"]]

# Build BM25 index
bm25 = BM25Okapi(tokenized_corpus)


### 8. Helper: top-k cosine neighbors for one story

In [826]:
def get_top_cosine_neighbors(story_index: int, similarity_matrix: np.ndarray, top_k: int = 5):
    """
    For a given story index, return the indices of the top-k most similar other stories
    based on cosine similarity.
    """
    sims = similarity_matrix[story_index].copy()

    # Exclude self-match
    sims[story_index] = -1

    # Get indices sorted by descending similarity
    top_indices = np.argsort(sims)[::-1][:top_k]

    # Return list of (neighbor_index, similarity_score)
    return [(idx, sims[idx]) for idx in top_indices]

### 9. Helper: top BM25 neighbors for one story

In [828]:
def get_top_bm25_neighbors(story_index: int, bm25_model: BM25Okapi, tokenized_corpus, top_k: int = 5):
    """
    For a given story index, return the indices of the top-k highest-scoring other stories
    based on BM25.
    """
    query_tokens = tokenized_corpus[story_index]

    # BM25 scores of this query story against all stories
    scores = bm25_model.get_scores(query_tokens)

    # Exclude self-match
    scores[story_index] = -1

    # Sort descending and keep top-k
    top_indices = np.argsort(scores)[::-1][:top_k]

    return [(idx, scores[idx]) for idx in top_indices]

### 10. Build enhanced candidate pairs for every story

In [830]:
def build_candidate_pairs(
    df: pd.DataFrame,
    similarity_matrix: np.ndarray,
    bm25_model: BM25Okapi,
    tokenized_corpus,
    cosine_top_k: int = 5,
    bm25_top_k: int = 5,
    bm25_min_score: float = 0.0
):
    """
    For each story:
    1. take the top-k cosine neighbors
    2. take the top-k BM25 neighbors
    3. add BM25 neighbors that are not already in the cosine set
    4. return all candidate pairs before duplicate removal

    Output format:
        source_story_id, target_story_id, source_story_text, target_story_text,
        candidate_source ('cosine' or 'bm25_enhanced'), score
    """
    all_pairs = []

    for i in range(len(df)):
        # ---- Step 1: cosine neighbors ----
        cosine_neighbors = get_top_cosine_neighbors(
            story_index=i,
            similarity_matrix=similarity_matrix,
            top_k=cosine_top_k
        )

        cosine_neighbor_ids = set()

        for neighbor_idx, sim_score in cosine_neighbors:
            cosine_neighbor_ids.add(neighbor_idx)

            all_pairs.append({
                "source_story_id": df.iloc[i]["story_id"],
                "target_story_id": df.iloc[neighbor_idx]["story_id"],
                "source_story_text": df.iloc[i]["story_text"],
                "target_story_text": df.iloc[neighbor_idx]["story_text"],
                "candidate_source": "cosine",
                "score": float(sim_score)
            })

        # ---- Step 2: BM25 neighbors ----
        bm25_neighbors = get_top_bm25_neighbors(
            story_index=i,
            bm25_model=bm25_model,
            tokenized_corpus=tokenized_corpus,
            top_k=bm25_top_k
        )

        # ---- Step 3: add only new BM25 neighbors ----
        for neighbor_idx, bm25_score in bm25_neighbors:
            if neighbor_idx in cosine_neighbor_ids:
                continue

            if bm25_score <= bm25_min_score:
                continue

            all_pairs.append({
                "source_story_id": df.iloc[i]["story_id"],
                "target_story_id": df.iloc[neighbor_idx]["story_id"],
                "source_story_text": df.iloc[i]["story_text"],
                "target_story_text": df.iloc[neighbor_idx]["story_text"],
                "candidate_source": "bm25_enhanced",
                "score": float(bm25_score)
            })

    return pd.DataFrame(all_pairs)

### 11. Run the candidate generation

In [832]:
candidate_pairs_raw = build_candidate_pairs(
    df=df,
    similarity_matrix=similarity_matrix,
    bm25_model=bm25,
    tokenized_corpus=tokenized_corpus,
    cosine_top_k=5,
    bm25_top_k=5,
    bm25_min_score=0.0   # you can raise this later if BM25 adds too much noise
)

print("Number of raw candidate pairs before duplicate removal:", len(candidate_pairs_raw))
candidate_pairs_raw

Number of raw candidate pairs before duplicate removal: 537


,source_story_id,target_story_id,source_story_text,target_story_text,candidate_source,score
0,0,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",cosine,0.900064
1,0,16,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a data packa...",cosine,0.689316
2,0,17,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to create a data packag...",cosine,0.651922
3,0,22,"As a Publisher, I want to publish a dataset, s...","As a publisher, I want to show the world how m...",cosine,0.630731
4,0,7,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to configure my client,...",cosine,0.620692
...,...,...,...,...,...,...
532,66,16,"As a Publisher, I want to have private data pa...","As a Publisher, I want to publish a data packa...",cosine,0.609872
533,66,7,"As a Publisher, I want to have private data pa...","As a Publisher, I want to configure my client,...",cosine,0.594492
534,66,0,"As a Publisher, I want to have private data pa...","As a Publisher, I want to publish a dataset, s...",bm25_enhanced,15.409986
535,66,24,"As a Publisher, I want to have private data pa...","As a Publisher, I want to preview a datapackag...",bm25_enhanced,14.571019


### 12. Remove duplicates: treat (A,B) and (B,A) as the same pair

In [834]:
def remove_symmetric_duplicates(pairs_df: pd.DataFrame):
    """
    Remove duplicate symmetric pairs:
    (A, B) and (B, A) are treated as the same pair.

    If the same pair was found by both cosine and BM25,
    we keep one row and preserve both sources.
    """
    df_copy = pairs_df.copy()

    # Create canonical pair key: smaller ID first, larger ID second
    df_copy["pair_id_1"] = df_copy.apply(
        lambda row: min(row["source_story_id"], row["target_story_id"]), axis=1
    )
    df_copy["pair_id_2"] = df_copy.apply(
        lambda row: max(row["source_story_id"], row["target_story_id"]), axis=1
    )

    # Aggregate duplicate rows
    grouped = (
        df_copy
        .groupby(["pair_id_1", "pair_id_2"], as_index=False)
        .agg({
            "source_story_text": "first",
            "target_story_text": "first",
            "candidate_source": lambda x: ", ".join(sorted(set(x))),
            "score": "max"
        })
    )

    return grouped

### 13. Create final unique candidate set

In [836]:
candidate_pairs_final = remove_symmetric_duplicates(candidate_pairs_raw)

print("Number of unique candidate pairs after duplicate removal:", len(candidate_pairs_final))
candidate_pairs_final.head(20)

Number of unique candidate pairs after duplicate removal: 354


,pair_id_1,pair_id_2,source_story_text,target_story_text,candidate_source,score
0,0,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",cosine,0.900064
1,0,7,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to configure my client,...",cosine,0.620692
2,0,16,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a data packa...",cosine,0.689316
3,0,17,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to create a data packag...",cosine,0.651922
4,0,21,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",bm25_enhanced,21.323694
5,0,22,"As a Publisher, I want to publish a dataset, s...","As a publisher, I want to show the world how m...",cosine,0.630731
6,0,23,"As a Publisher, I want to publish a dataset, s...","As a consumer, I want to view the data package...",bm25_enhanced,20.805535
7,0,46,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a Datapackage at...",bm25_enhanced,21.001159
8,0,58,"As an owner, I want to view all the people in ...","As a Publisher, I want to publish a dataset, s...",bm25_enhanced,16.857372
9,0,66,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to have private data pa...",bm25_enhanced,17.276006


------------

In [844]:
# Build lookup dictionary from story_id to story_text
story_lookup = dict(zip(df["story_id"], df["story_text"]))

candidate_pairs_final["story_1_text"] = candidate_pairs_final["pair_id_1"].map(story_lookup)
candidate_pairs_final["story_2_text"] = candidate_pairs_final["pair_id_2"].map(story_lookup)

# Reorder columns for clarity
candidate_pairs_final = candidate_pairs_final[
    ["pair_id_1", "pair_id_2", "story_1_text", "story_2_text", "candidate_source", "score"]
]

print(len(candidate_pairs_final))
candidate_pairs_final.head(20)

354


,pair_id_1,pair_id_2,story_1_text,story_2_text,candidate_source,score
0,0,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",cosine,0.900064
1,0,7,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to configure my client,...",cosine,0.620692
2,0,16,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a data packa...",cosine,0.689316
3,0,17,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to create a data packag...",cosine,0.651922
4,0,21,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",bm25_enhanced,21.323694
5,0,22,"As a Publisher, I want to publish a dataset, s...","As a publisher, I want to show the world how m...",cosine,0.630731
6,0,23,"As a Publisher, I want to publish a dataset, s...","As a consumer, I want to view the data package...",bm25_enhanced,20.805535
7,0,46,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a Datapackage at...",bm25_enhanced,21.001159
8,0,58,"As a Publisher, I want to publish a dataset, s...","As an owner, I want to view all the people in ...",bm25_enhanced,16.857372
9,0,66,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to have private data pa...",bm25_enhanced,17.276006


In [849]:
# Count raw candidate sources
print(candidate_pairs_raw["candidate_source"].value_counts())
print('\n')
# Count final candidate sources
print(candidate_pairs_final["candidate_source"].value_counts())

candidate_source
cosine           335
bm25_enhanced    202
Name: count, dtype: int64


candidate_source
cosine                   206
bm25_enhanced            121
bm25_enhanced, cosine     27
Name: count, dtype: int64


In [851]:
for i, row in candidate_pairs_final.iterrows():
    
    print("=" * 100)
    print(f"PAIR {i}")
    print(f"Story IDs: {row['pair_id_1']}  <->  {row['pair_id_2']}")
    print(f"Source: {row['candidate_source']}")
    print(f"Score: {row['score']:.4f}")
    
    print("\nStory 1:")
    print(row["story_1_text"])
    
    print("\nStory 2:")
    print(row["story_2_text"])
    
    print("\n")

PAIR 0
Story IDs: 0  <->  1
Source: cosine
Score: 0.9001

Story 1:
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.

Story 2:
As a Publisher, I want to publish a dataset, so that I can share the dataset publicly with everyone.


PAIR 1
Story IDs: 0  <->  7
Source: cosine
Score: 0.6207

Story 1:
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.

Story 2:
As a Publisher, I want to configure my client, so that I can start publishing data packages.


PAIR 2
Story IDs: 0  <->  16
Source: cosine
Score: 0.6893

Story 1:
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few people.

Story 2:
As a Publisher, I want to publish a data package in the UI, so that that it is available and published.


PAIR 3
Story IDs: 0  <->  17
Source: cosine
Score: 0.6519

Story 1:
As a Publisher, I want to publish a dataset, so that I can view just the dataset with a few peop

In [853]:
candidate_pairs_final.to_csv(r'preprocessed_pairs\g14-datahub.csv',index=False)

# --------- Preprocessing OF TRANSFORMATIONS--------------------

In [8]:
import json
from pathlib import Path

txt_folder = Path("prompts")

txt_files = [
    txt_folder / "paraphrase_fixed.txt",
    txt_folder / "policy.txt",
    txt_folder / "prerequisite.txt",
    txt_folder / "state_condition.txt",
]

for file_path in txt_files:
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            json.load(f)
        print(f"[OK] {file_path}")
    except json.JSONDecodeError as e:
        print(f"\n[ERROR] {file_path}")
        print(f"Message: {e.msg}")
        print(f"Line: {e.lineno}, Column: {e.colno}, Char: {e.pos}")

        with open(file_path, "r", encoding="utf-8") as f:
            lines = f.readlines()

        start = max(e.lineno - 3, 0)
        end = min(e.lineno + 2, len(lines))

        print("Context:")
        for i in range(start, end):
            prefix = ">>" if i + 1 == e.lineno else "  "
            print(f"{prefix} {i+1}: {lines[i].rstrip()}")

[OK] prompts\paraphrase_fixed.txt
[OK] prompts\policy.txt
[OK] prompts\prerequisite.txt
[OK] prompts\state_condition.txt


In [12]:
import pandas as pd
import json
from pathlib import Path

# your original pairs dataframe
dff = pd.read_csv(r'preprocessed_pairs\g14-datahub.csv')

# folder containing the 4 txt files
txt_folder = Path(r"prompts")   # <-- change this

txt_files = [
    txt_folder / "paraphrase_fixed.txt",
    txt_folder / "policy.txt",
    txt_folder / "prerequisite.txt",
    txt_folder / "state_condition.txt",
]

# -----------------------------
# 1) Build a lookup from story text -> pair_id
# -----------------------------
story1_lookup = (
    dff[["pair_id_1", "story_1_text"]]
    .drop_duplicates(subset=["story_1_text"])
    .rename(columns={"pair_id_1": "pair_id", "story_1_text": "source_story"})
)

story2_lookup = (
    dff[["pair_id_2", "story_2_text"]]
    .drop_duplicates(subset=["story_2_text"])
    .rename(columns={"pair_id_2": "pair_id", "story_2_text": "source_story"})
)

story_lookup = (
    pd.concat([story1_lookup, story2_lookup], ignore_index=True)
    .drop_duplicates(subset=["source_story", "pair_id"])
)

# optional: if the same source_story appears with multiple pair_ids, inspect them
duplicate_sources = story_lookup[story_lookup.duplicated("source_story", keep=False)].sort_values("source_story")
print("Stories with multiple pair_ids in lookup:")
print(duplicate_sources if not duplicate_sources.empty else "None")


# -----------------------------
# 2) Read all txt JSON files and extract rows
# -----------------------------
all_rows = []

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    items = data.get("items", [])
    
    for item in items:
        all_rows.append({
            "file_name": file_path.stem,
            "story_id": item.get("story_id"),
            "source_story": item.get("source_story"),
            "generated_story": item.get("generated_story"),
            "transformation": item.get("transformation"),
        })

transformed_df = pd.DataFrame(all_rows)

print("Extracted rows:", len(transformed_df))
display(transformed_df.head())

Stories with multiple pair_ids in lookup:
None
Extracted rows: 96


,file_name,story_id,source_story,generated_story,transformation
0,paraphrase_fixed,1,"As a Visitor, I want to sign up via github or ...","As a Visitor, I want to register using GitHub ...",PARAPHRASE_STORY_CREATION
1,paraphrase_fixed,2,"As a Publisher, I want to know what do next af...","As a Publisher, I want guidance on what to do ...",PARAPHRASE_STORY_CREATION
2,paraphrase_fixed,3,"As an Admin, I want to invite someone to join ...","As an Admin, I want to send an invitation for ...",PARAPHRASE_STORY_CREATION
3,paraphrase_fixed,4,"As a Publisher, I want to import my data packa...","As a Publisher, I want to upload my data packa...",PARAPHRASE_STORY_CREATION
4,paraphrase_fixed,5,"As a Publisher, I want to use a publish comman...","As a Publisher, I want to update a data packag...",PARAPHRASE_STORY_CREATION


In [14]:
transformed_df

,file_name,story_id,source_story,generated_story,transformation
0,paraphrase_fixed,1,"As a Visitor, I want to sign up via github or ...","As a Visitor, I want to register using GitHub ...",PARAPHRASE_STORY_CREATION
1,paraphrase_fixed,2,"As a Publisher, I want to know what do next af...","As a Publisher, I want guidance on what to do ...",PARAPHRASE_STORY_CREATION
2,paraphrase_fixed,3,"As an Admin, I want to invite someone to join ...","As an Admin, I want to send an invitation for ...",PARAPHRASE_STORY_CREATION
3,paraphrase_fixed,4,"As a Publisher, I want to import my data packa...","As a Publisher, I want to upload my data packa...",PARAPHRASE_STORY_CREATION
4,paraphrase_fixed,5,"As a Publisher, I want to use a publish comman...","As a Publisher, I want to update a data packag...",PARAPHRASE_STORY_CREATION
...,...,...,...,...,...
91,state_condition,3,"As a Consumer, I want to load a Data Package f...","As a Consumer, I want to load a locally cached...",STATE_OR_CONDITION_STORY_CREATION
92,state_condition,4,As a Data Analyst I want to download a data pa...,"As a Data Analyst, I want to download an archi...",STATE_OR_CONDITION_STORY_CREATION
93,state_condition,5,"As a Data Analyst, I want to update previously...","As a Data Analyst, I want to update a previous...",STATE_OR_CONDITION_STORY_CREATION
94,state_condition,6,"As a Publisher, I want to unpublish a data pac...","As a Publisher, I want to unpublish a publishe...",STATE_OR_CONDITION_STORY_CREATION


In [15]:
final_df = transformed_df.merge(
    story_lookup,
    on="source_story",
    how="left"
)

# arrange columns
final_df = final_df[
    ["pair_id", "source_story", "generated_story", "transformation", "file_name", "story_id"]
]

print(final_df.head())
print("Missing pair_id rows:", final_df["pair_id"].isna().sum())

   pair_id                                       source_story  \
0        3  As a Visitor, I want to sign up via github or ...   
1        4  As a Publisher, I want to know what do next af...   
2        5  As an Admin, I want to invite someone to join ...   
3        6  As a Publisher, I want to import my data packa...   
4        8  As a Publisher, I want to use a publish comman...   

                                     generated_story  \
0  As a Visitor, I want to register using GitHub ...   
1  As a Publisher, I want guidance on what to do ...   
2  As an Admin, I want to send an invitation for ...   
3  As a Publisher, I want to upload my data packa...   
4  As a Publisher, I want to update a data packag...   

              transformation         file_name story_id  
0  PARAPHRASE_STORY_CREATION  paraphrase_fixed        1  
1  PARAPHRASE_STORY_CREATION  paraphrase_fixed        2  
2  PARAPHRASE_STORY_CREATION  paraphrase_fixed        3  
3  PARAPHRASE_STORY_CREATION  paraphrase

In [18]:
final_df

,pair_id,source_story,generated_story,transformation,file_name,story_id
0,3,"As a Visitor, I want to sign up via github or ...","As a Visitor, I want to register using GitHub ...",PARAPHRASE_STORY_CREATION,paraphrase_fixed,1
1,4,"As a Publisher, I want to know what do next af...","As a Publisher, I want guidance on what to do ...",PARAPHRASE_STORY_CREATION,paraphrase_fixed,2
2,5,"As an Admin, I want to invite someone to join ...","As an Admin, I want to send an invitation for ...",PARAPHRASE_STORY_CREATION,paraphrase_fixed,3
3,6,"As a Publisher, I want to import my data packa...","As a Publisher, I want to upload my data packa...",PARAPHRASE_STORY_CREATION,paraphrase_fixed,4
4,8,"As a Publisher, I want to use a publish comman...","As a Publisher, I want to update a data packag...",PARAPHRASE_STORY_CREATION,paraphrase_fixed,5
...,...,...,...,...,...,...
91,32,"As a Consumer, I want to load a Data Package f...","As a Consumer, I want to load a locally cached...",STATE_OR_CONDITION_STORY_CREATION,state_condition,3
92,33,As a Data Analyst I want to download a data pa...,"As a Data Analyst, I want to download an archi...",STATE_OR_CONDITION_STORY_CREATION,state_condition,4
93,34,"As a Data Analyst, I want to update previously...","As a Data Analyst, I want to update a previous...",STATE_OR_CONDITION_STORY_CREATION,state_condition,5
94,9,"As a Publisher, I want to unpublish a data pac...","As a Publisher, I want to unpublish a publishe...",STATE_OR_CONDITION_STORY_CREATION,state_condition,6


In [17]:
final_df_simple = final_df[["pair_id", "source_story", "generated_story"]]
display(final_df_simple.head())

,pair_id,source_story,generated_story
0,3,"As a Visitor, I want to sign up via github or ...","As a Visitor, I want to register using GitHub ..."
1,4,"As a Publisher, I want to know what do next af...","As a Publisher, I want guidance on what to do ..."
2,5,"As an Admin, I want to invite someone to join ...","As an Admin, I want to send an invitation for ..."
3,6,"As a Publisher, I want to import my data packa...","As a Publisher, I want to upload my data packa..."
4,8,"As a Publisher, I want to use a publish comman...","As a Publisher, I want to update a data packag..."


In [32]:
# cheching that after removing pairs with all NONE, how may none values and how many pairs do we still have left
final_list = [3, 4, 5, 6, 8, 9, 13, 22, 25, 26, 30, 31, 32, 33, 34, 35, 37, 38, 43, 45, 46, 48, 49, 54, 57, 60, 61, 62]
dff_clean = dff[
    ~dff["pair_id_1"].isin(final_list) &
    ~dff["pair_id_2"].isin(final_list)
].copy()
dff_clean

,pair_id_1,pair_id_2,story_1_text,story_2_text,candidate_source,score
0,0,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",cosine,0.900064
1,0,7,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to configure my client,...",cosine,0.620692
2,0,16,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a data packa...",cosine,0.689316
3,0,17,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to create a data packag...",cosine,0.651922
4,0,21,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",bm25_enhanced,21.323694
...,...,...,...,...,...,...
346,58,64,"As an owner, I want to view all the people in ...","As a Publisher, I want to know if this site ha...",bm25_enhanced,24.548384
348,59,63,"As an owner, I want to make a user an owner, s...","As an Admin, I want to have a pricing plan and...",bm25_enhanced,16.504612
351,63,64,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to know if this site ha...",cosine,0.527958
352,63,65,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to sign up for a given ...",cosine,0.618391


In [22]:
dff

,pair_id_1,pair_id_2,story_1_text,story_2_text,candidate_source,score
0,0,1,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a dataset, s...",cosine,0.900064
1,0,7,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to configure my client,...",cosine,0.620692
2,0,16,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to publish a data packa...",cosine,0.689316
3,0,17,"As a Publisher, I want to publish a dataset, s...","As a Publisher, I want to create a data packag...",cosine,0.651922
4,0,21,"As a Publisher, I want to publish a dataset, s...","As a Consumer, I want to view a data package o...",bm25_enhanced,21.323694
...,...,...,...,...,...,...
349,61,62,"As an Admin, I want to set key configuration p...","As an Admin, I want to see key metrics about u...","bm25_enhanced, cosine",17.204144
350,62,63,"As an Admin, I want to see key metrics about u...","As an Admin, I want to have a pricing plan and...","bm25_enhanced, cosine",16.570193
351,63,64,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to know if this site ha...",cosine,0.527958
352,63,65,"As an Admin, I want to have a pricing plan and...","As a Publisher, I want to sign up for a given ...",cosine,0.618391


In [ ]:
# final_df_simple.to_csv("final_transformed_stories_with_pair_id.csv", index=False)